# DR05 — Generative Quality, Diversity & Efficiency

DR05 evaluates generative quality, diversity, and computational efficiency. Every displayed value is loaded from frozen canonical artifacts; this notebook does not rerun the final evaluation, metrics, training, or sampling. Quality and diversity are not conditioning adherence: conditioning and compositional adherence are evaluated separately in DR04.

## 1. Scope and canonical evidence

This is a thin, read-only evidence layer. It verifies artifact identities before parsing them, then presents the frozen DR05 results. The artifact registry below is deliberately explicit and repository-relative.

In [1]:
from __future__ import annotations

import hashlib
from html import escape
import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

from IPython.display import HTML, display


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'tiny_avatar_diffusion').is_dir():
            return candidate
    raise RuntimeError('Run from the repository root or the notebooks directory.')


ROOT = find_project_root()

ARTIFACTS = {
    'quality summary': ('results/runs/dr05_final_quality_v1/final_quality_summary.json', '20d6b20a59908443c3d963e935d06f0eccc907b813dce8ce242d1a2da7b27592'),
    'quality provenance addendum': ('results/runs/dr05_final_quality_v1/dr05_i25_provenance_addendum.json', '8d5cae9cf7485c160c4c402645f2cb0e0a0d0453a972ee9abd2944ea069c8bee'),
    'diversity pair records': ('results/runs/dr05_final_diversity_v1/pair_records.jsonl', 'd6aa49a7de6c7838033dce3bf343e202a080ac4f094c8d6d4a4523d0223673bc'),
    'diversity prompt records': ('results/runs/dr05_final_diversity_v1/prompt_records.jsonl', '0bd808994e2064485cf366102f5190d9a1267d360f751a9faeb4f89ec09d5a6c'),
    'diversity summary': ('results/runs/dr05_final_diversity_v1/diversity_summary.json', '85ccd22f592a1ac91a0760895e6b5100b5110b53c2843b2e4076dbc469348e7b'),
    'parameter count': ('results/runs/dr05_final_efficiency_v1/parameter_count_summary.json', '0d8593c59c5608873b2c7d67240af18024bc4e6427724cd397aed684add24deb'),
    'latency summary': ('results/runs/dr05_final_efficiency_v1/latency_summary.json', 'd706c13bb6872971168757ec41a62145bc92e807b2f55673de0c06a5ac9144d4'),
    'memory conditional B1': ('results/runs/dr05_final_efficiency_v1/conditional_b1.json', '302b3b64e0be3a56c8c7e980bb6d33d13db1edd91aa01e3c49911185228f5d89'),
    'memory unconditional B1': ('results/runs/dr05_final_efficiency_v1/unconditional_b1.json', '7cdd8a645476ce3bb1b5f5b79aa1126cf3fdbe9a4fc928087ce771d346da6c25'),
    'memory conditional B16': ('results/runs/dr05_final_efficiency_v1/conditional_b16.json', 'f870591096618207f10db21ee09cb04451b87508dd6c9706ba1e432084f62a35'),
    'memory unconditional B16': ('results/runs/dr05_final_efficiency_v1/unconditional_b16.json', '6e8d476bd36c2302905c51b54a75873eb837ee6bfa218d4d60865eb03bccb03d'),
    'DR05 protocol': ('data/processed/dr05_final_eval_v1/dr05_final_protocol.json', '706a191fff8f2fba1b263919385a6e55ad4579a5285a189cab5acf20ae38549f'),
    'execution bindings': ('data/processed/dr05_final_eval_v1/dr05_execution_bindings.json', '603a9121f980b769621b628a0cf393dda17dd10a5e6aeca6e6e94bcc3e671ebf'),
}


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_json(relative_path: str) -> dict:
    return json.loads((ROOT / relative_path).read_text(encoding='utf-8'))


def show_table(rows: list[dict]) -> list[dict]:
    rows = list(rows)
    columns = list(rows[0]) if rows else []
    header = ''.join(f'<th>{escape(str(column))}</th>' for column in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(row.get(column, "")))}</td>' for column in columns) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))
    return rows


## 2. Canonical artifact integrity

In [2]:
integrity_rows = []
for logical_name, (relative_path, expected_sha256) in ARTIFACTS.items():
    path = ROOT / relative_path
    observed_sha256 = sha256(path)
    assert observed_sha256 == expected_sha256, f'canonical artifact hash mismatch: {relative_path}'
    integrity_rows.append({'artifact': logical_name, 'path': relative_path, 'SHA256': observed_sha256, 'verified': True})

integrity_table = show_table(integrity_rows)


artifact,path,SHA256,verified
quality summary,results/runs/dr05_final_quality_v1/final_quality_summary.json,20d6b20a59908443c3d963e935d06f0eccc907b813dce8ce242d1a2da7b27592,True
quality provenance addendum,results/runs/dr05_final_quality_v1/dr05_i25_provenance_addendum.json,8d5cae9cf7485c160c4c402645f2cb0e0a0d0453a972ee9abd2944ea069c8bee,True
diversity pair records,results/runs/dr05_final_diversity_v1/pair_records.jsonl,d6aa49a7de6c7838033dce3bf343e202a080ac4f094c8d6d4a4523d0223673bc,True
diversity prompt records,results/runs/dr05_final_diversity_v1/prompt_records.jsonl,0bd808994e2064485cf366102f5190d9a1267d360f751a9faeb4f89ec09d5a6c,True
diversity summary,results/runs/dr05_final_diversity_v1/diversity_summary.json,85ccd22f592a1ac91a0760895e6b5100b5110b53c2843b2e4076dbc469348e7b,True
parameter count,results/runs/dr05_final_efficiency_v1/parameter_count_summary.json,0d8593c59c5608873b2c7d67240af18024bc4e6427724cd397aed684add24deb,True
latency summary,results/runs/dr05_final_efficiency_v1/latency_summary.json,d706c13bb6872971168757ec41a62145bc92e807b2f55673de0c06a5ac9144d4,True
memory conditional B1,results/runs/dr05_final_efficiency_v1/conditional_b1.json,302b3b64e0be3a56c8c7e980bb6d33d13db1edd91aa01e3c49911185228f5d89,True
memory unconditional B1,results/runs/dr05_final_efficiency_v1/unconditional_b1.json,7cdd8a645476ce3bb1b5f5b79aa1126cf3fdbe9a4fc928087ce771d346da6c25,True
memory conditional B16,results/runs/dr05_final_efficiency_v1/conditional_b16.json,f870591096618207f10db21ee09cb04451b87508dd6c9706ba1e432084f62a35,True


## 3. Generative quality

KID and FID describe similarity to the frozen real-image reference pools. On these frozen splits the conditional generator has lower KID and FID than the unconditional generator. This does not establish prompt adherence or compositional generalization.

In [3]:
quality = load_json(ARTIFACTS['quality summary'][0])
comparisons = quality['state']['phase3']['metrics']
quality_expected = {
    'conditional:ID': (0.019295956939458847, 49.37773171781788),
    'conditional:OOD': (0.019467167556285858, 58.702624366029426),
    'conditional:GLOBAL': (0.018365884199738503, 40.99542135937443),
    'unconditional:ID': (0.02868834137916565, 58.08331071638449),
    'unconditional:OOD': (0.04378482326865196, 83.18393889462249),
    'unconditional:GLOBAL': (0.03191326931118965, 53.19797715139188),
}
quality_rows = []
for key, (expected_kid, expected_fid) in quality_expected.items():
    record = comparisons[key]['result']
    assert record['kid'] == expected_kid and record['fid'] == expected_fid
    model, split = key.split(':')
    quality_rows.append({'model': model, 'split': split, 'KID': record['kid'], 'FID': record['fid']})
quality_table = show_table(quality_rows)


model,split,KID,FID
conditional,ID,0.019295956939458847,49.37773171781788
conditional,OOD,0.019467167556285858,58.702624366029426
conditional,GLOBAL,0.018365884199738503,40.99542135937443
unconditional,ID,0.02868834137916565,58.08331071638449
unconditional,OOD,0.04378482326865196,83.18393889462249
unconditional,GLOBAL,0.03191326931118965,53.19797715139188


## 4. Diversity across seeds

LPIPS is summarized over the frozen prompt/seed population. Higher LPIPS is not automatically better: diversity must be interpreted jointly with quality and conditioning adherence.

In [4]:
diversity = load_json(ARTIFACTS['diversity summary'][0])
aggregates = diversity['aggregates']
diversity_keys = {
    'ID': ('ID_prompt_macro_lpips', 0.036734598668868114),
    'OOD': ('OOD_prompt_macro_lpips', 0.03531421060534194),
    'GLOBAL': ('GLOBAL_prompt_macro_lpips', 0.03617095261191328),
    'OOD1': ('OOD1_prompt_macro_lpips', 0.03856944183539599),
    'OOD2': ('OOD2_prompt_macro_lpips', 0.03931253510061652),
    'OOD3': ('OOD3_prompt_macro_lpips', 0.0381747223418206),
    'OOD4': ('OOD4_prompt_macro_lpips', 0.02520014314353466),
    'equal-family OOD macro': ('OOD_equal_family_macro_lpips', 0.03531421060534194),
}
diversity_rows = []
for scope, (key, expected_value) in diversity_keys.items():
    assert aggregates[key] == expected_value
    ci = diversity.get('bootstrap', {}).get('scopes', {}).get(scope, {}).get('percentile_95')
    diversity_rows.append({'scope': scope, 'mean LPIPS': aggregates[key], '95% percentile CI': ci})
diversity_table = show_table(diversity_rows)


scope,mean LPIPS,95% percentile CI
ID,0.036734598668868114,"[0.035663168653424546, 0.0377540886825093]"
OOD,0.03531421060534194,"[0.03380818334654323, 0.03677022509957897]"
GLOBAL,0.03617095261191328,None
OOD1,0.03856944183539599,"[0.03648689214910847, 0.04025766636379994]"
OOD2,0.03931253510061652,"[0.03819422987508588, 0.040482802818831994]"
OOD3,0.0381747223418206,"[0.03521693052998744, 0.04104346009152942]"
OOD4,0.02520014314353466,"[0.02432950114184059, 0.026219176538661126]"
equal-family OOD macro,0.03531421060534194,None


## 5. Parameter count

In [5]:
parameters = load_json(ARTIFACTS['parameter count'][0])
conditional_parameters = parameters['conditional']
parameter_rows = [
    ('Unconditional U-Net', conditional_parameters['unconditional_backbone']),
    ('Cross-attention', conditional_parameters['cross_attention']),
    ('Conditional U-Net', conditional_parameters['conditional_unet']),
    ('Text encoder', conditional_parameters['text_encoder']),
    ('Full conditional generator', conditional_parameters['full_system']),
    ('Evaluator (separate instrument)', parameters['evaluation_instrument']),
]
parameter_table = show_table([{'component': name, **counts} for name, counts in parameter_rows])
assert parameters['unconditional']['total_parameters'] == 4_595_779
assert parameters['evaluation_instrument']['explicitly_excluded_from_generator_counts'] is True
assert all(item['residual'] == 0 and item['pass'] for item in parameters['anti_double_count']['equations'].values())
print('Evaluator excluded from generator counts:', parameters['evaluation_instrument']['explicitly_excluded_from_generator_counts'])


component,total_parameters,trainable_parameters
Unconditional U-Net,4595779,4595779
Cross-attention,25152,25152
Conditional U-Net,4620931,4620931
Text encoder,102976,102976
Full conditional generator,4723907,4723907
Evaluator (separate instrument),293064,293064


Evaluator excluded from generator counts: True


## 6. Sampling latency

B1 is the primary headline; B16 is a secondary engineering-throughput diagnostic.

In [6]:
latency = load_json(ARTIFACTS['latency summary'][0])
latency_by_key = {(item['model_type'], item['batch_size']): item for item in latency['conditions']}
conditional_b1 = latency_by_key[('conditional', 1)]
unconditional_b1 = latency_by_key[('unconditional', 1)]
conditional_b16 = latency_by_key[('conditional', 16)]
unconditional_b16 = latency_by_key[('unconditional', 16)]
assert conditional_b1['role'] == unconditional_b1['role'] == 'PRIMARY_HEADLINE'
assert conditional_b16['role'] == unconditional_b16['role'] == 'SECONDARY_ENGINEERING_THROUGHPUT'
assert latency['timing']['warmups'] == 3 and latency['timing']['measured_runs'] == 10
assert all(item['precision'] == 'FP32' and item['resolution'] == [32, 32] and item['diffusion_steps'] == 1000 and item['sampler'] == 'ancestral_ddpm_fixed_small' for item in latency['conditions'])
b1_conditional = conditional_b1['statistics']['median_seconds_per_image']
b1_unconditional = unconditional_b1['statistics']['median_seconds_per_image']
overhead_seconds = b1_conditional - b1_unconditional
overhead_percent = overhead_seconds / b1_unconditional * 100
assert abs(b1_conditional - 8.7465763500) < 1e-9 and abs(b1_unconditional - 7.5675747500) < 1e-9
latency_table = show_table([
    {'role': 'PRIMARY_HEADLINE', 'model': 'conditional', 'batch': 1, 'median s/img': b1_conditional, 'images/s': conditional_b1['statistics']['images_per_second']},
    {'role': 'PRIMARY_HEADLINE', 'model': 'unconditional', 'batch': 1, 'median s/img': b1_unconditional, 'images/s': unconditional_b1['statistics']['images_per_second']},
    {'role': 'SECONDARY_ENGINEERING_THROUGHPUT', 'model': 'conditional', 'batch': 16, 'median s/img': conditional_b16['statistics']['median_seconds_per_image'], 'images/s': conditional_b16['statistics']['images_per_second']},
    {'role': 'SECONDARY_ENGINEERING_THROUGHPUT', 'model': 'unconditional', 'batch': 16, 'median s/img': unconditional_b16['statistics']['median_seconds_per_image'], 'images/s': unconditional_b16['statistics']['images_per_second']},
])
print(f'B1 conditional overhead: {overhead_seconds:.10f} s/img ({overhead_percent:.10f}%)')


role,model,batch,median s/img,images/s
PRIMARY_HEADLINE,conditional,1,8.746576350007672,0.11433044885032334
PRIMARY_HEADLINE,unconditional,1,7.567574749991763,0.1321427317253904
SECONDARY_ENGINEERING_THROUGHPUT,conditional,16,0.6305745937497704,1.5858552024010468
SECONDARY_ENGINEERING_THROUGHPUT,unconditional,16,0.5521374531244874,1.811143211425173


B1 conditional overhead: 1.1790016000 s/img (15.5796492135%)


## 7. Sampling memory

B1 is the primary memory headline; B16 is an engineering diagnostic. PyTorch peak allocated memory is not total system video-memory use.

In [7]:
memory_rows = []
memory_expected = {
    'memory conditional B1': (29.81884765625, 40.0),
    'memory unconditional B1': (29.30859375, 40.0),
    'memory conditional B16': (77.73095703125, 124.0),
    'memory unconditional B16': (77.0185546875, 112.0),
}
for name, (expected_allocated, expected_reserved) in memory_expected.items():
    record = load_json(ARTIFACTS[name][0])
    assert record['peak_allocated_mib'] == expected_allocated
    assert record['peak_reserved_mib'] == expected_reserved
    assert record['precision'] == 'FP32' and record['resolution'] == [32, 32] and record['diffusion_steps'] == 1000
    memory_rows.append({'model': record['model_type'], 'batch': record['batch_size'], 'peak allocated MiB': record['peak_allocated_mib'], 'peak reserved MiB': record['peak_reserved_mib']})
memory_table = show_table(sorted(memory_rows, key=lambda row: (row['batch'], row['model'])))


model,batch,peak allocated MiB,peak reserved MiB
conditional,1,29.81884765625,40.0
unconditional,1,29.30859375,40.0
conditional,16,77.73095703125,124.0
unconditional,16,77.0185546875,112.0


## 8. Quality is not conditioning

Quality measures visual-distribution similarity, and diversity measures variation across seeds. Neither directly proves prompt adherence. Conditioning adherence and compositionality are evaluated separately in `04_conditioning_compositional_evaluation.ipynb`; DR05 does not duplicate those DR04 tables.

## 9. DR05 canonical closure summary

In [8]:
closure_summary = show_table([
    {'category': 'Quality', 'headline result': 'Frozen KID/FID table across ID, OOD, and GLOBAL', 'canonical artifact': ARTIFACTS['quality summary'][0]},
    {'category': 'Diversity', 'headline result': f"GLOBAL mean LPIPS = {aggregates['GLOBAL_prompt_macro_lpips']:.6f}", 'canonical artifact': ARTIFACTS['diversity summary'][0]},
    {'category': 'Conditional parameters', 'headline result': '4,723,907', 'canonical artifact': ARTIFACTS['parameter count'][0]},
    {'category': 'Unconditional parameters', 'headline result': '4,595,779', 'canonical artifact': ARTIFACTS['parameter count'][0]},
    {'category': 'B1 conditional latency', 'headline result': f'{b1_conditional:.10f} s/img', 'canonical artifact': ARTIFACTS['latency summary'][0]},
    {'category': 'B1 unconditional latency', 'headline result': f'{b1_unconditional:.10f} s/img', 'canonical artifact': ARTIFACTS['latency summary'][0]},
    {'category': 'B1 latency overhead', 'headline result': f'{overhead_seconds:.10f} s/img ({overhead_percent:.10f}%)', 'canonical artifact': ARTIFACTS['latency summary'][0]},
    {'category': 'B1 conditional peak memory', 'headline result': '29.81884765625 MiB allocated', 'canonical artifact': ARTIFACTS['memory conditional B1'][0]},
    {'category': 'B1 unconditional peak memory', 'headline result': '29.30859375 MiB allocated', 'canonical artifact': ARTIFACTS['memory unconditional B1'][0]},
])


category,headline result,canonical artifact
Quality,"Frozen KID/FID table across ID, OOD, and GLOBAL",results/runs/dr05_final_quality_v1/final_quality_summary.json
Diversity,GLOBAL mean LPIPS = 0.036171,results/runs/dr05_final_diversity_v1/diversity_summary.json
Conditional parameters,"4,723,907",results/runs/dr05_final_efficiency_v1/parameter_count_summary.json
Unconditional parameters,"4,595,779",results/runs/dr05_final_efficiency_v1/parameter_count_summary.json
B1 conditional latency,8.7465763500 s/img,results/runs/dr05_final_efficiency_v1/latency_summary.json
B1 unconditional latency,7.5675747500 s/img,results/runs/dr05_final_efficiency_v1/latency_summary.json
B1 latency overhead,1.1790016000 s/img (15.5796492135%),results/runs/dr05_final_efficiency_v1/latency_summary.json
B1 conditional peak memory,29.81884765625 MiB allocated,results/runs/dr05_final_efficiency_v1/conditional_b1.json
B1 unconditional peak memory,29.30859375 MiB allocated,results/runs/dr05_final_efficiency_v1/unconditional_b1.json


## 10. Reproducibility and environment

The environment below is descriptive only. Artifact parsing is the reproducibility contract for this notebook; no scientific computation is repeated.

In [9]:
environment = {
    'Python': sys.version.split()[0],
    'platform': platform.platform(),
    'nbformat': metadata.version('nbformat'),
    'notebook artifact root': str(ROOT.relative_to(ROOT)),
}
show_table([{'field': field, 'value': value} for field, value in environment.items()])
print('scientific recomputation performed: NO')
print('training performed: NO')
print('sampling performed: NO')
print('canonical artifacts modified: NO')


field,value
Python,3.12.4
platform,Windows-11-10.0.26300-SP0
nbformat,5.11.1
notebook artifact root,.


scientific recomputation performed: NO
training performed: NO
sampling performed: NO
canonical artifacts modified: NO
